# Mem0 — Phase 1: Memory Layers & Basic Storage

**Goal of this notebook:** build the simplest possible working version of Mem0's storage layer —
just enough to store a "memory" and pull it back out when it's relevant.

We're not doing extraction, decay, or conflict handling yet — that's later phases.
Right now we just want **one fact in → one fact out** working end to end.

**Running example (from the Mem0 paper):** an aircraft maintenance engineer who works on
Boeing 737 hydraulic systems, talking to an assistant across multiple sessions.


## 1. The four memory layers

Mem0 splits memories into four "layers" based on how long they should live:

| Layer | What it holds | Example |
|---|---|---|
| `user` | long-term facts about the person | "Works with Boeing 737" |
| `session` | facts that only matter in the current chat | "Currently troubleshooting a hydraulic leak" |
| `agent` | things the assistant learned about *how* to behave | "Aviation questions need extra accuracy" |
| `cross_session` | knowledge that should carry forward and compound over time | general lessons learned across many users |

For Phase 1, this is just a **label we attach to each memory** — a string. We're not building
separate storage for each layer yet, we're just tagging memories so we *could* filter by layer later.


In [1]:
# This is just a plain Python list — it's here so you can see the four layer names
# written down in one place. We'll use these exact strings as tags on our memories.
MEMORY_LAYERS = ["user", "session", "agent", "cross_session"]

print(MEMORY_LAYERS)

['user', 'session', 'agent', 'cross_session']


## 2. Install the vector database

We're using **ChromaDB** — a free, local vector database. "Vector database" sounds fancy,
but all it does is:
1. Store some text
2. Convert that text into a list of numbers (an "embedding")
3. Let you search by "which stored texts are numerically closest to my query"

Run the cell below once. `-q` just means "quiet" (don't print all the install logs).
(`--break-system-packages` is only needed on some Linux setups — harmless to leave in
if you're on Colab or a normal virtual environment.)


In [2]:
!pip install chromadb -q --break-system-packages

## 3. Turning text into numbers (embeddings) — the simple way

Normally, people use a pretrained model (like OpenAI's or a HuggingFace model) to turn text
into embeddings. Those models need to be downloaded or called over the internet, which can be
slow or fail in restricted environments.

For this class, we'll write our **own tiny embedding function** instead. It won't be smart —
it just counts which words appear — but it's enough to prove the *mechanism* works, and it
runs instantly with no downloads. Later, in a real project, you'd swap this out for a proper
embedding model without changing anything else in the pipeline.

**How it works, line by line:**
1. Make an empty vector of zeros (64 slots)
2. Split the input text into lowercase words
3. For each word, hash it into one of the 64 slots and add 1 to that slot
4. The result is a vector where "similar sentences" (sharing words) end up numerically close


In [3]:
from chromadb import Documents, EmbeddingFunction, Embeddings
import hashlib

class SimpleWordVectorEmbedding(EmbeddingFunction):
    # vector_size = how many numbers represent each piece of text
    def __init__(self, vector_size=64):
        self.vector_size = vector_size

    # Chroma calls this function automatically whenever it needs to embed text
    def __call__(self, input):
        vectors = []                              # will hold one vector per input text
        for text in input:                        # loop over each piece of text
            vector = [0.0] * self.vector_size      # start with a vector of all zeros
            words = text.lower().split()           # lowercase, then split into words
            for word in words:                     # loop over each word
                # hash the word into a number, then squeeze it into range 0-63
                slot = int(hashlib.md5(word.encode()).hexdigest(), 16) % self.vector_size
                vector[slot] += 1.0                # count that word by bumping its slot
            vectors.append(vector)                 # save this text's finished vector
        return vectors                             # give Chroma the list of vectors

# quick sanity check: two sentences sharing words should look similar
embedder = SimpleWordVectorEmbedding()
print(embedder(["Boeing 737 hydraulic system", "hydraulic system pressure"]))

[array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32), array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)]


## 4. Create the memory store

Now we create a Chroma "collection" — think of it like a table in a database, but for memories.
We tell it to use our `SimpleWordVectorEmbedding` whenever it needs to turn text into numbers.


In [4]:
import chromadb

# Client() with no arguments creates an in-memory database.
# Everything disappears when the notebook restarts -- fine for learning.
client = chromadb.Client()

# create_collection makes a new named "table" for our memories
memory_store = client.create_collection(
    name="mem0_demo",
    embedding_function=embedder,
)

print("Memory store created:", memory_store.name)

Memory store created: mem0_demo


## 5. `add_memory` — writing a memory

This is a small wrapper function around Chroma's `.add()` method. Every memory we store needs:
- `text` — the actual fact
- `layer` — which of the 4 layers it belongs to
- `user_id` — who this memory belongs to
- `memory_id` — a unique ID (Chroma requires one per record)

We're keeping this deliberately simple — no validation, no error handling yet. If you pass
in bad data, it'll just fail loudly, which is fine while we're learning.


In [5]:
def add_memory(text, layer, user_id, memory_id):
    memory_store.add(
        documents=[text],                 # the fact itself
        metadatas=[{                      # extra info attached to the fact
            "layer": layer,
            "user_id": user_id,
        }],
        ids=[memory_id],                  # every memory needs a unique ID
    )
    print(f"Stored [{layer}]: {text}")

## 6. `search_memory` — reading memories back

Given a query, this returns the most relevant stored memories for a specific user.


In [6]:
def search_memory(query, user_id, n_results=3):
    results = memory_store.query(
        query_texts=[query],              # what we're searching for
        n_results=n_results,              # how many matches to return
        where={"user_id": user_id},       # only search this user's memories
    )
    return results["documents"][0]        # the [0] is because we only sent one query

## 7. Try it: the aircraft engineer, session by session

Now let's replay the paper's example. In later phases, an "extraction engine" will pull these
facts out of raw conversation automatically. For now, we're adding them by hand, just to prove
the store/retrieve loop works.


### Session 1 — asks about hydraulic pressure specs

In [7]:
add_memory("User works with Boeing 737", layer="user", user_id="eng_01", memory_id="m1")
add_memory("User is interested in hydraulic systems", layer="user", user_id="eng_01", memory_id="m2")

Stored [user]: User works with Boeing 737
Stored [user]: User is interested in hydraulic systems


### Session 2 — mentions "our fleet" (we infer they manage multiple aircraft)

In [8]:
add_memory("User manages multiple aircraft", layer="user", user_id="eng_01", memory_id="m3")

Stored [user]: User manages multiple aircraft


### Session 3 — asks about troubleshooting, not just theory

In [9]:
add_memory("User needs practical repair guidance, not just specs", layer="user", user_id="eng_01", memory_id="m4")
add_memory("Currently troubleshooting a hydraulic leak", layer="session", user_id="eng_01", memory_id="m5")

Stored [user]: User needs practical repair guidance, not just specs
Stored [session]: Currently troubleshooting a hydraulic leak


## 8. The payoff: Session 4

A new session starts. The user asks a short question with zero context.
If memory is working, the system should still "know" they mean the 737's hydraulic backup system.


In [10]:
matches = search_memory("what's the backup system?", user_id="eng_01")

print("Retrieved memories for this query:")
for m in matches:
    print(" -", m)

Retrieved memories for this query:
 - User manages multiple aircraft
 - User works with Boeing 737
 - User is interested in hydraulic systems


If this printed back the Boeing 737 / hydraulic / troubleshooting facts, the core loop works:
**write a memory → embed it → store it → retrieve it later by meaning, not exact words.**

That's the whole engine Mem0 is built on. Everything in later phases (real extraction from
conversation text, importance scoring, decay, conflict resolution) is refinement on top of
this same write/read loop — not a replacement for it.

**Next up — Phase 2:** replacing the manual `add_memory(...)` calls above with an actual
extraction step that reads raw conversation text and pulls facts out automatically.
